# Module 7, Section 1: Online Evals, Alerts and Routing

Module 4 Section 3 ends with four loops. The fourth is the hill-climbing loop: production traces drive improvements to the agent. This module runs that loop on one day of production traffic of the TechHub deep support agent.

**An online evaluator is a function that LangSmith runs on each new trace and that writes a score on it.** It needs no reference answer: it judges the run on its own.

In this section:
1. The data: one day of TechHub production traffic
2. Three signals (errors, user scores, reading traces) and what each one misses
3. An online evaluator that finds wrong return-window answers
4. Filters and sampling: what you pay for and what you miss
5. An alert threshold that the background errors cannot reach
6. Routing flagged runs to an annotation queue for review

## Setup

This section calls no model. `uv sync` is all you need, and a LangSmith key is optional. Without a key, every cell runs on the data on disk, and the steps in LangSmith are described instead of done.

| Where you are | What to set |
|---|---|
| No LangSmith key | Nothing. The Setup cell prints one line that starts with ⏭️ and the notebook runs locally |
| Project-based workspace: the control at the top left of LangSmith shows the LangSmith logo | `LANGSMITH_API_KEY` in `.env`. The traces go to the project `techhub-deep-agent-production`; set `M7_PROJECT` to use another name |
| Agent-based workspace: the control at the top left shows your workspace name | `LANGSMITH_API_KEY` and `TECHHUB_AGENT_ID` in `.env`. The traces go to that agent's Production environment; `TECHHUB_AGENT_ENVIRONMENT` picks another one |

With a key, some cells also write to your workspace: the seeded day, its scores and a review queue. Each of those cells starts with a flag such as `UPLOAD_SEED`. Set it to `False` to skip that write. If a write fails, the cell prints one line and the notebook continues locally.

In [ ]:
import json, os, sys, warnings
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(REPO_ROOT))
# Load .env before any workshop import: config.py picks the models from it at import time
load_dotenv(REPO_ROOT / ".env")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=r".*is deprecated and will be removed after", category=DeprecationWarning)

from utils.langsmith_links import setup_langsmith
from utils.trace_seeding import describe, load_seed, print_trace
from utils.trace_upload import resolve_destination

LANGSMITH_AVAILABLE = setup_langsmith()            # one probe at most, one printed line
DEST = resolve_destination(LANGSMITH_AVAILABLE)    # project name, project id, navigation sentence
DATA = REPO_ROOT / "workshop_modules" / "module_7" / "data"
seed = load_seed()                                 # timestamps shifted so the newest run is 5 minutes old

## 1. One day of TechHub production traffic

TechHub put the deep support agent into production in Workshop 9. Yesterday evening the team shipped version 1.5.0. You are on call this morning with one day of traffic: 200 conversations and 260 customer turns. Five things in that day are wrong or wasteful, and nobody has told the team.

This module follows the order an on-call engineer works in. You detect problems with online evals and alerts (this section), understand them with Insights (Section 2), fix them with Engine (Section 3) and prevent them with masking and Gateway policies (Section 4). One silent failure runs through Sections 1 to 3: the agent gives some customers the wrong return window.

The data is synthetic. Every run has the tag `seed:m7` and the metadata `synthetic: true`. The customer and order ids are real ids from `data/structured/techhub.db`.

One customer turn is one trace. When the main agent asks the order specialist for one lookup, the trace has 14 runs (each `model > ChatOpenAI` is two runs: the graph node and the model call inside it):

```text
deep_support_agent              the root run: one customer turn
├── model > ChatOpenAI          the main agent plans
├── tools > task                it hands the question to a specialist
│   └── order-specialist
│       ├── model > ChatOpenAI
│       ├── tools > get_order_status
│       └── model > ChatOpenAI
└── model > ChatOpenAI          the main agent writes the reply
```

The seeded traces carry these 14 runs. A live trace of the same agent has about 95, because each middleware step adds a run of its own.

In [ ]:
summary = describe(seed)
print()
print_trace(seed, "t-0042")

### Put the same day in your workspace (optional)

With a key, the next cell uploads the 260 traces and their 170 user scores. That is 260 of the 5,000 traces a month that a free organization without a payment method gets. To find them:

> In LangSmith, open **Tracing**. If the control at the top left shows your workspace name (an agent-based workspace), select the agent **TechHub support** in the agent switcher in the top bar, then set the environment control to **Production**. If it shows the LangSmith logo (a project-based workspace), open the tracing project `techhub-deep-agent-production` (or the name you set in `M7_PROJECT`). Set the time range to at least the last 2 days.

The Setup cell printed the same path for your workspace, after "Seeded traffic goes to:".

The upload writes every user score with `extend_trace_retention=False`. With the SDK's default, a score moves its whole trace from base retention (14 days on LangSmith Cloud) to extended retention (180 days), which costs more.

Running the upload twice is safe. It first looks for traces tagged `seed:m7` from the last 7 days and sends nothing if it finds them.

In [ ]:
from utils.trace_upload import upload_seed

UPLOAD_SEED = LANGSMITH_AVAILABLE  # set to False to keep the seeded day out of your workspace
if UPLOAD_SEED:
    SEEDED = upload_seed(seed, DEST)
else:
    SEEDED = False
    print("⏭️  Skipped: nothing was uploaded. Every cell below analyses the same 260 traces locally.")

## 2. Three signals, and what each one misses

| Signal | What it finds | What it misses |
|---|---|---|
| Error status of a run | Crashes and timeouts | Answers that are wrong and raise no error |
| User feedback (thumbs up or down) | Answers customers dislike | Wrong answers that customers like, and conversations nobody rates (30 of the 200 here) |
| Reading traces by hand | Anything you notice | Everything you have no time to read: 260 traces a day here, far more in a real product |

In [ ]:
from datetime import datetime, timezone
from utils.monitoring_local import error_windows

windows = error_windows(seed)  # the whole seed, so the failing tool inside each trace can be named
burst = windows["burst"]
now = datetime.now(timezone.utc)
for row in windows["errors"]:
    minutes_ago = (now - row["start_time"]).total_seconds() / 60
    where = "burst" if row["in_burst"] else "background"
    print(f"{row['start_time']:%H:%M:%S} UTC  {minutes_ago:5.0f} min ago  {row['trace_key']}  {row['failing_tool'] or '-':<20}  {where}")
print()
print(
    f"{len(windows['errors'])} errored traces. {burst['errors']} of them inside {burst['span_seconds'] / 60:.0f} minutes, "
    f"about {(now - burst['first']).total_seconds() / 60:.0f} minutes ago, tool {', '.join(burst['failing_tools'])}."
)
print("The same traces in the LangSmith search bar: status:error")

In [ ]:
import re
from collections import Counter
from utils.monitoring_checks import check
from utils.trace_seeding import load_labels, root_runs, runs_by_trace, trace_key, user_scores

roots = root_runs(seed)
by_trace = runs_by_trace(seed)
scores = user_scores(seed)  # trace key of a conversation's last turn -> 1 (thumbs-up) or 0 (thumbs-down)
version_of = {trace_key(run): run["extra"]["metadata"]["app_version"] for run in roots}
conversations = Counter({run["extra"]["metadata"]["thread_id"]: version_of[trace_key(run)] for run in roots}.values())

down = [key for key, score in scores.items() if score == 0]
down_by_version = Counter(version_of[key] for key in down)
print(f"{len(down)} thumbs-down: " + ", ".join(f"{100 * down_by_version[v] / conversations[v]:.1f}% on {v}" for v in sorted(conversations)) + ".")

NOT_FOUND = re.compile(r"Order \S+ not found\.")


def tool_results(key):
    return [str(((run["outputs"] or {}).get("output") or {}).get("content", "")) for run in by_trace[key] if run["run_type"] == "tool"]


after_not_found = [key for key in down if any(NOT_FOUND.fullmatch(text) for text in tool_results(key))]
after_error = [key for key in down if any(run["error"] for run in by_trace[key])]
print(f"{len(after_not_found)} follow an 'Order ... not found.' tool result. {len(after_error)} follow an errored run.")

# The label file holds the ground truth for every conversation. It exists only because the data is
# synthetic: a real project has no such file. Here it says which conversations got the wrong return window.
labels = load_labels()
wrong_window = [c for c in labels["conversations"] if c["incident"] == "A"]
down_on_wrong = sum(c["user_score"] == 0 for c in wrong_window)
print(f"The {len(wrong_window)} conversations with the wrong return window got {down_on_wrong} thumbs-down.")

ok, message = check("s1_signals", {
    "errored_root_runs": len(windows["errors"]),
    "thumbs_down_conversations": len(down),
    "thumbs_down_after_order_not_found": len(after_not_found),
    "thumbs_down_on_wrong_return_window": down_on_wrong,
})
print(message)

The error count found the burst in `get_order_status`. The thumbs-down found the orders that the agent reported as missing. Neither signal finds a confident wrong answer: the 11 customers who got the wrong return window gave 9 thumbs-up and no thumbs-down. Reading all 260 traces would find them, but you cannot do that every day. You need a check that knows the policy and runs on every trace: an online evaluator.

## 3. An online evaluator is a function of one run

<div align="center">
    <img src="../../images/online_eval_process.png" width="800">
</div>

A **code evaluator** is a Python function that LangSmith runs once for each run that passes the evaluator's filter. LangSmith passes the run as a dictionary and stores each key of the returned dictionary as feedback on that run.

| Rule | What it means here |
|---|---|
| Name and arguments | `perform_eval(run, example=None)`. LangSmith passes a second argument when you test the evaluator on a dataset example, so the function must accept one |
| Input | `run["inputs"]["messages"]` ends with the customer's message. `run["outputs"]["messages"]` is the conversation so far and ends with the reply. A failed run has empty outputs |
| Output | A dictionary of feedback key to score, such as `{"wrong_return_window": True}` |
| Packages | The standard library and a few packages that LangSmith lists. The function has no internet access |

In this evaluator `True` means the evaluator found a problem. A high share of `True` scores is bad news.

TechHub's return policy, from `data/documents/policies/return_policy.md`:

```text
**Unopened Electronics**
- 30 days from delivery date
- All original packaging and seals must be intact
- Full refund to original payment method

**Opened Electronics**
- 14 days from delivery date
```

> 💡 **Try it first:** write `perform_eval` in `workshop_modules/module_7/exercises/return_window_evaluator.py`, then check it from the repository root with `uv run python workshop_modules/module_7/exercises/check.py s1_evaluator`. The next two cells show a naive version and the reference version.

In [ ]:
from utils.monitoring_local import apply_evaluator, flagged_trace_keys
from utils.trace_seeding import question_text, reply_text, trace_runs


def naive_eval(run, example=None):
    """Flag every reply that mentions 30 days."""
    return {"wrong_return_window": "30 days" in reply_text(run)}


naive_flagged = flagged_trace_keys(apply_evaluator(naive_eval, roots))
condition = {key: (c.get("details") or {}).get("item_condition") for c in labels["conversations"] for key in c["trace_keys"]}
unopened = [key for key in naive_flagged if condition.get(key) == "unopened"]
print(f"Flagged {len(naive_flagged)} of {len(roots)}. {len(unopened)} of them are correct answers about unopened items.")

opened = next(key for key in naive_flagged if condition.get(key) == "opened")
for key in (unopened[0], unopened[1], opened):
    root = trace_runs(seed, key)[0]
    print()
    print(f"{key} (the item is {condition[key]})")
    print("  Customer:", question_text(root))
    print("  Agent:   ", reply_text(root))

In [ ]:
from evaluators import WRONG_RETURN_WINDOW_SOURCE, wrong_return_window_evaluator

rows = apply_evaluator(wrong_return_window_evaluator, roots)
flagged = flagged_trace_keys(rows)
print(f"Flagged {len(flagged)} of {len(roots)}.")
ok, message = check("s1_evaluator", wrong_return_window_evaluator)
print(message)

versions = sorted({version_of[key] for key in flagged})
with_policy_search = sum(any(run["name"] == "search_policy_docs" for run in by_trace[key]) for key in flagged)
up = sum(scores.get(key) == 1 for key in flagged)
thumbs_down = sum(scores.get(key) == 0 for key in flagged)
unrated = sum(key not in scores for key in flagged)
print(f"All on app_version {', '.join(versions)}. Traces with a search_policy_docs run: {with_policy_search}.")
print(f"Their user scores: {up} up, {thumbs_down} down, {unrated} without a score.")

In [ ]:
from utils.trace_upload import write_scores

# The documented replacement for the deprecated evaluate_run(): run the evaluator yourself and log
# each result as feedback with create_feedback().
WRITE_SCORES = SEEDED  # set to False to keep the scores out of your workspace
if WRITE_SCORES:
    write_scores(seed, DEST, wrong_return_window_evaluator)
    print("To see the flagged runs, paste this into the LangSmith search bar: feedback.wrong_return_window.score:>0.5")
else:
    print(f"⏭️  Skipped. On our runs: {len(flagged)} runs carry wrong_return_window = 1.")

### 3.1 Attach it to live traffic

The notebook scored yesterday's traces itself. In LangSmith, an evaluator scores new traces as they arrive. With a key, do these steps yourself. Without one, the facilitator shows them. The labels come from the LangSmith docs and can differ a little in your workspace.

| Step | In the UI |
|---|---|
| 1. Open your TechHub traces | **Tracing** > agent **TechHub support** > **Production**; in a project-based workspace, **Tracing** > project `techhub-deep-agent-production`. Set the time range to at least the last 2 days |
| 2. Start an evaluator | The **Evaluators** tab (**Online Evaluators** in an agent-based workspace) > **+ Evaluator** > **Code Evaluator** under **Create from scratch** |
| 3. Name it | `wrong_return_window`, the same as the key the function returns. The docs say the evaluator's name is also a feedback name, so one name for both avoids two keys |
| 4. Filter | **Run Name** is `deep_support_agent`, so only root runs are scored |
| 5. Sampling rate | 1.0: every matching run |
| 6. Past runs | Leave **Apply to past runs** off. The notebook has already scored yesterday |
| 7. Code | Paste the function that the next cell prints, click **Test Code**, then **Save** |

In [ ]:
from utils.trace_seeding import root_runs
from utils.trace_upload import build_live_traces, send_live_traces

print(WRONG_RETURN_WINDOW_SOURCE)

SEND_LIVE_TRACES = SEEDED  # set to False to send no fresh traces
if SEND_LIVE_TRACES:
    send_live_traces("quality", 4, DEST)
else:
    live_runs, _ = build_live_traces("quality", 4)  # built here and not sent
    live_flagged = flagged_trace_keys(apply_evaluator(wrong_return_window_evaluator, root_runs(live_runs)))
    print(f"⏭️  Skipped. With the evaluator attached, {len(live_flagged)} of the 4 new traces are scored wrong_return_window = 1.")

In [ ]:
from utils.trace_upload import live_scores

if SEEDED:
    live_scores(DEST)  # counts the fresh traces that already carry the score; it does not wait
else:
    print("⏭️  Skipped: no fresh traces were sent.")

## 4. Filters and sampling: what you pay for and what you miss

Two settings decide which runs an evaluator scores: a filter and a sampling rate. A code evaluator costs no model tokens. An LLM-as-judge evaluator pays for every run it scores, and with retention extension on, every scored trace moves to extended retention.

| Lever | Effect on cost | Effect on coverage |
|---|---|---|
| Filter | Only matching runs are scored | Problems outside the filter are missed |
| Sampling rate | A random share of the matching runs is scored | Problems in the runs that are not drawn are missed |
| Weekly evaluator spend limit | LLM-as-judge evaluators stop at a set amount per week | Nothing is scored after the limit |
| Retention | Retention extension is on by default for new evaluators: each scored trace moves to extended retention | None |

Filter strings in the documented filter syntax:

| Filter | Selects |
|---|---|
| `eq(is_root, true)` | Every root run: one per customer turn |
| `and(eq(is_root, true), eq(metadata_key, "app_version"), eq(metadata_value, "1.5.0"))` | Root runs of release 1.5.0 |
| `eq(name, "product-policy-specialist")` | The product-policy specialist's run inside each trace |

To pick out the model calls of one agent, filter on the metadata key `lc_agent_name`. The run names `model` and `ChatOpenAI` occur in every agent, and the name of the model run changes with the provider.

The next cell compares three configurations for the evaluator of §3. The 25% sample is a fixed draw, so the exercise has one answer. LangSmith draws each run on its own, so a real 25% rate scores about a quarter of the runs, a different quarter every time.

In [ ]:
from statistics import median
from utils.monitoring_local import FILTERS, sample, select

caught = set(flagged)


def scored_and_caught(runs):
    return {"scored": len(runs), "caught": sum(trace_key(run) in caught for run in runs)}


configurations = {
    "all_root_runs": select("root_runs", seed),
    "app_version_1_5_0": select("root_runs_on_1_5_0", seed),
    "sample_25_percent": sample(roots, rate=0.25),
}
names = {"all_root_runs": "All root runs", "app_version_1_5_0": "Filter app_version 1.5.0", "sample_25_percent": "25% sample"}
for name, runs in configurations.items():
    result = scored_and_caught(runs)
    print(f"{names[name] + ':':<27} {result['scored']:>3} scored, {result['caught']:>2} caught")
print()
print("Filter strings:")
for name in ("root_runs", "root_runs_on_1_5_0"):
    print(f"  {FILTERS[name].filter}")

draws = [scored_and_caught(sample(roots, rate=0.25, seed=draw))["caught"] for draw in range(100, 120)]
print()
print(f"20 other 25% draws catch between {min(draws)} and {max(draws)} of the {len(caught)} (median {median(draws):g}).")

ok, message = check("s1_filter_sample", {name: scored_and_caught(runs) for name, runs in configurations.items()})
print(message)

## 5. From scores to an alert

An alert watches one metric of a tracing project (or of one agent environment) and notifies you when the metric crosses a threshold.

| Part | Choices |
|---|---|
| Metric | Run Count, Cost, Errors, Feedback Score, Latency |
| Threshold | A number and a comparison such as `>=` |
| Aggregation window | 5 or 15 minutes |
| Channel | Slack, PagerDuty, Dynatrace or a webhook |

A threshold that is too low pages someone for background errors. A threshold that is too high misses a real burst.

> 💡 **Try it first:** in the error list of §2, what is the largest number of background errors in any 5 minutes? Which threshold can they never reach?

In [ ]:
from utils.monitoring_local import error_bars

print(f"Errors per {windows['minutes']}-minute window (windows with no error are left out):")
for line in error_bars(windows):
    print(line)
print("These windows start on the clock, so one burst can fall into two of them. The counts below hold wherever a window starts.")
print()
background = windows["background"]["max_in_any_window"]
print(
    f"Background errors: at most {background} in any {windows['minutes']} minutes. "
    f"Burst: {burst['errors']} in {burst['span_seconds'] / 60:.0f} minutes. "
    f"Smallest threshold the background cannot reach: {windows['smallest_safe_threshold']}."
)
ok, message = check("s1_alert", {
    "background_max_in_5_minutes": background,
    "burst_errors": burst["errors"],
    "threshold": windows["smallest_safe_threshold"],
    "failing_tool": burst["failing_tools"][0],
})
print(message)

### The same alert in LangSmith

| Step | In the UI |
|---|---|
| 1. Open your TechHub traces | **Tracing** > agent **TechHub support** > **Production**; in a project-based workspace, **Tracing** > project `techhub-deep-agent-production`. Set the time range to at least the last 2 days |
| 2. Start an alert | The **Alerts** icon in the top right-hand corner. The docs also call it **+ Alert** and **Alerts > Create Alert**; use the one you see |
| 3. Metric | **Errors**, as a count. It counts every run with an error status, and each failed trace of the seeded day holds 6 errored runs. To count failed traces, count root runs only (`eq(is_root, true)` as an API filter). Whether the alert form offers a root-run condition is UNVERIFIED |
| 4. Condition | Threshold 3 failed traces (errored root runs), **Aggregation Window** 5 minutes |
| 5. Preview | Read the historical preview: it marks the windows that would have fired |
| 6. Channel | A webhook or Slack. Click **Send Test Alert** (**Send Test Notification** for Slack) |

Two facts to know before you rely on an alert:

- **Send Test Alert** does not check the receiver's response. The UI reports success even when the receiver returned an error, so look at the receiver's own log.
- LangSmith sends the body you configure as written, with no template substitution, and adds 12 keys at the top level. The next cell prints them.

The seeded burst is close to an hour old (§2 printed its age), so it cannot trip a live 5-minute window. Whether the preview shows back-dated runs is UNVERIFIED. To see a real firing, set `TRIP_ALERT = True` in the next cell: it sends 5 errored runs dated now.

In [ ]:
payload = json.loads((DATA / "alert_webhook_payload.json").read_text(encoding="utf-8"))
keys = [key for key in payload if not key.startswith("_")]
print(f"The {len(keys)} keys LangSmith adds to an alert webhook body (the values are examples):")
for key in keys:
    print(f"  {key}: {payload[key]!r}")
print()

TRIP_ALERT = False  # set to True (with a key, after the upload) to send 5 errored runs dated now
if TRIP_ALERT and SEEDED:
    send_live_traces("errors", 5, DEST)
else:
    print("⏭️  Not sent. With a key and TRIP_ALERT = True, this cell sends 5 errored runs dated now.")

## 6. Route what was flagged to people

<div align="center">
    <img src="../../images/data_flywheel.png" width="800">
</div>

A score marks a run as wrong. A person still has to confirm it and write the correct answer, which then becomes a test case. An **automation rule** sends runs to that person. It has an item type, a filter, a sampling rate and an action, such as adding the run to an annotation queue.

> 💡 **Try it first:** which filter selects the root runs that the evaluator scored `wrong_return_window` = 1? The filter syntax has the fields `feedback_key` and `feedback_score`.

In [ ]:
from utils.trace_upload import send_to_queue

route = FILTERS["flagged_wrong_return_window"]
selected = select("flagged_wrong_return_window", seed, rows)  # rows: the evaluator's scores from §3
print("Filter:", route.filter)
print(f"{len(selected)} runs selected.")
ok, message = check("s1_routing", selected)
print(message)

CREATE_QUEUE = SEEDED  # set to False to create no queue in your workspace
if CREATE_QUEUE:
    send_to_queue(DEST, [trace_key(run) for run in selected])
else:
    print("⏭️  Skipped: no queue was created, because nothing was uploaded.")

### Automate it for new traffic, and review one item

The cell above routed yesterday's runs once. An automation does the same for every new run:

| Step | In the UI |
|---|---|
| 1. Open your TechHub traces | **Tracing** > agent **TechHub support** > **Production**; in a project-based workspace, **Tracing** > project `techhub-deep-agent-production`. Set the time range to at least the last 2 days |
| 2. Start an automation | **+ New** > **New Automation**, then a name |
| 3. Item type | **Runs** |
| 4. Filter | Feedback key `wrong_return_window`, score above 0.5 |
| 5. Action | **Add to Annotation Queue**, queue `techhub-deep-agent-return-window-review` |
| 6. Past runs | Leave **Apply to Past Runs** off |

Then review one item, as in Module 3:

| Step | In the UI |
|---|---|
| 1. Open the queue | **Annotation Queues** in the left navigation, then `techhub-deep-agent-return-window-review`. A queue lists its items whatever their age, so it has no time range |
| 2. Validate | Read the customer's message and the reply. The customer opened the product and was told 30 days |
| 3. Add feedback | Score the run as the queue's instructions say |
| 4. Correct the output | Turn on **Classic Trace View** to edit the output, and change the window to 14 days |
| 5. Keep it as a test case | **Add to Dataset** > `techhub-deep-agent-regressions` |

In [ ]:
from utils.monitoring_rest import code_evaluator_rule, create_rule_once, queue_rule
from utils.trace_upload import QUEUE_NAME, new_client

# Optional: the evaluator of §3.1 and the automation above as two REST requests. Sending them is
# UNVERIFIED against a live workspace, so the flag is off. Printing a body sends nothing.
MONITORING_AS_CODE = False
project_id = DEST.project_id or "<project id>"
evaluator_body = code_evaluator_rule(project_id=project_id, code=WRONG_RETURN_WINDOW_SOURCE)
shown = {**evaluator_body, "code_evaluators": [{"code": "<WRONG_RETURN_WINDOW_SOURCE>", "language": "python"}]}
print("POST /api/v1/runs/rules  (attach the evaluator)")
print(json.dumps(shown, indent=2))
print("POST /api/v1/runs/rules  (route flagged runs to the queue)")
print(json.dumps(queue_rule(project_id=project_id, queue_id="<queue id>"), indent=2))
print()

if MONITORING_AS_CODE and SEEDED:
    try:
        client = new_client()
        bodies = [code_evaluator_rule(project_id=DEST.project_id, code=WRONG_RETURN_WINDOW_SOURCE)]
        queue = next(iter(client.list_annotation_queues(name=QUEUE_NAME)), None)
        if queue is None:
            print(f"No queue named {QUEUE_NAME}: run the routing cell of §6 first. Only the evaluator is sent.")
        else:
            bodies.append(queue_rule(project_id=DEST.project_id, queue_id=str(queue.id)))
        for body in bodies:
            rule = create_rule_once(client, body)  # lists the project's rules first and skips a duplicate name
            print(f"{body['display_name']}: " + ("created." if rule else "a rule of this name exists, nothing was sent."))
    except Exception as exc:  # noqa: BLE001 - any failure falls back to the UI steps
        print(f"⏭️  The request failed ({type(exc).__name__}). Use the UI steps of §3.1 and §6 instead.")
else:
    print("⏭️  Not sent. With a key and MONITORING_AS_CODE = True, this cell sends both requests.")

### What you did in code, and where it is in LangSmith

| Step in this notebook | Same step in the UI |
|---|---|
| `upload_seed(seed, DEST)` | Traces arrive from your application. Nothing to click |
| `apply_evaluator(...)`, `write_scores(...)` | An online evaluator attached on the **Evaluators** tab (§3.1) |
| `select("root_runs_on_1_5_0", seed)`, `sample(roots, rate=0.25)` | The evaluator's filter and sampling rate |
| `error_windows(seed)` | An alert on **Errors** with a 5-minute window (§5) |
| `send_to_queue(DEST, ...)` | An automation with the action **Add to Annotation Queue** (§6) |

This section only described LLM-as-judge and thread evaluators. Both need a model provider key stored in the workspace. A thread evaluator also needs a `messages` key in the top-level inputs and outputs of every trace, and it runs only after a thread has been idle for the project's idle time (10 minutes by default, at least 2).

## Recap

- The error count and the thumbs-down missed the 11 wrong return-window answers: they raised no error and got no thumbs-down.
- A code evaluator is a function `perform_eval(run, example=None)` that returns feedback. It needs no model, so it can score every trace.
- The filter on app_version 1.5.0 scored 104 runs and caught all 11. The 25% sample scored 65 and caught 3.
- An alert with a threshold of 3 failed traces (errored root runs) in 5 minutes stays quiet for the background (at most 2) and fires on the burst of 8.
- An automation on the evaluator's feedback sends flagged runs to a review queue, where people turn them into test cases.

**Next:** Section 2 reads an Insights report of the same day and checks one of its claims with this evaluator.